# EXIST 2021 - Feature extraction train <a class="anchor" id="feat-train"></a>

    ?LVARO FAUBEL SANCHIS
    CLARA MART? TORREGROSA

#####  Table of contents :
- [Requiered functions](#functions)
 * [Required libraries and configuration](#libraries)
 * [Pre-processing](#pre-pro)
 * [Feature extraction](#feat)
 * [Models](#models)
     
- [EXIST Task](#exist)
 * [Data load](#data-load)
 * [Modeling](#model)
     - [Baseline](#baseline)
         - [Task 1](#b-t1)
         - [Task 2](#b-t2)
     - [Tunning parameters](#tunning)
         - [Task 1](#tp-t1)
         - [Task 2](#tp-t2)
     - [Ensemble methods](#ensemble)
         - [Task 1](#e-t1)
         - [Task 2](#e-t2)




### Requiered functions  <a class="anchor" id="functions"></a>

#### Required libraries and configuration  <a class="anchor" id="libraries"></a>

In [ ]:
# Data
import pandas as pd
import numpy as np

# Pre-processing
import preprocessor as p
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer, SnowballStemmer

# Feature extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from gensim.models import word2vec
import gensim.downloader as api

# Models
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn import svm
from sklearn import tree

#### Pre-processing  <a class="anchor" id="pre-pro"></a>

In [ ]:
def preprocess_tweets(l_tweets, l_langs, keep_hastags = False):
    tweets_res = []
    tweets_stem = []
    tw_tknz = TweetTokenizer()
    stem_sp = SnowballStemmer('spanish')
    lem_en = WordNetLemmatizer()
    stopwords_es_en = set(stopwords.words(['english', 'spanish']))
    p.set_options(p.OPT.URL, p.OPT.MENTION, p.OPT.RESERVED,  
                  p.OPT.SMILEY, p.OPT.HASHTAG)
    if keep_hastags:
        p.set_options(p.OPT.URL, p.OPT.MENTION, p.OPT.RESERVED,
                      p.OPT.SMILEY)
    
    def clean(tweet):
        clean_tw = p.clean(tweet.lower())
        clean_numbers =  re.sub(r'(?:\d+)(?:\w)*', ' ', clean_tw)
        separe_compound = re.sub(r'-', ' ', clean_numbers)
        clean_punct = re.sub(r'[^a-z???????#\s]', ' ', separe_compound)
        clean_spaces = re.sub(' +', ' ', clean_punct)
        return clean_spaces.strip()

    def tokenize(tweet_clean):
        return tw_tknz.tokenize(tweet_clean)
    
    def del_stopwords(tweet_token, l_stopwords):
        return [w for w in tweet_token if w not in l_stopwords]
    
    def do_stemming(tweet_token_nstop, stemer):
        return [stemer.stem(w) for w in tweet_token_nstop]

    def do_lemmatization(tweet_token_nstop, lemmatizer):
        return [lemmatizer.lemmatize(w) for w in tweet_token_nstop]
    
    for tw, lang in zip(l_tweets, l_langs):
        tw_clean = clean(tw)
        tw_token = tokenize(tw_clean)
        tw_nstop = del_stopwords(tw_token, stopwords_es_en)
        if lang == 'es':
            tweets_stem.append(do_stemming(tw_nstop, stem_sp))
        elif lang == 'en':
            tweets_stem.append(do_lemmatization(tw_nstop, lem_en))
        tweets_res.append(tw_nstop)
        
    return tweets_res, tweets_stem

#### Feature extraction <a class="anchor" id="feat"></a>

*  Bag of words

In [ ]:
def create_bag_of_words(l_docs):
    vectorizer = CountVectorizer(tokenizer=lambda doc: doc, lowercase=False)
    bag_of_words = vectorizer.fit_transform(l_docs)
    #df_bag = pd.DataFrame(bag_of_words.toarray(), columns=vectorizer.get_feature_names())
    return vectorizer, bag_of_words

*  N-grams

In [ ]:
def create_ngrams(l_docs, analyzer_type = 'word', ngram = 1):
    if analyzer_type != 'word':
        l_docs = [' '.join(tw) for tw in l_docs]
    vectorizer = CountVectorizer(tokenizer=lambda doc: doc, lowercase=False, 
                             analyzer=analyzer_type, ngram_range=(ngram,ngram))    
    counts = vectorizer.fit_transform(l_docs)
    #df_counts = pd.DataFrame(counts.toarray(), columns=vectorizer.get_feature_names())
    return vectorizer, counts

* TF-IDF

In [ ]:
def codify_tfidf_vec(l_docs, idf=True, analyzer_type = 'word', ngram=1):
    if analyzer_type != 'word':
        l_docs = [' '.join(tw) for tw in l_docs]
    tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda doc: doc, lowercase=False, 
                                       smooth_idf=True, use_idf=idf,
                                       analyzer=analyzer_type,
                                       ngram_range=(ngram,ngram))
    counts = tfidf_vectorizer.fit_transform(l_docs)
    #df_counts = pd.DataFrame(counts.toarray(), columns=tfidf_vectorizer.get_feature_names())
    return tfidf_vectorizer, counts

* Word Embeddings

In [ ]:
def get_vectors_embeddings(model_pretrained, model_data, l_tweets):
    vectors = []
    for i,tw in enumerate(l_tweets):
        tw_vector = []
        for w in tw:
            try:
                tw_vector.append(model_pretrained.get_vector(w))
            except KeyError:
                try:
                    tw_vector.append(model_data.wv.get_vector(w))
                except KeyError:
                    pass
        sum_vec = sum(tw_vector)
        if type(sum_vec) == int:
            vectors.append(np.zeros(200))
        else:
            vectors.append(sum_vec/len(tw))
    return np.array(vectors)

#### Models <a class="anchor" id="models"></a>

* Cross validation

In [ ]:
def crossval_kfold(X, y, model, k, shuffle = True):
    accuracys = []
    f1 = []
    kfs = KFold(k, shuffle = shuffle)
    for train_index, test_index in kfs.split(X):
        X_train, y_train = X[train_index], y[train_index]
        X_test, y_test = X[test_index], y[test_index]
        clf = model.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        accuracys.append(accuracy_score(y_test, y_pred))
        f1.append(f1_score(y_test, y_pred, average='macro'))
    return accuracys, f1

---

# EXIST Task <a class="anchor" id="exist"></a>

## Data load <a class="anchor" id="data-load"></a>

#### Train data

In [ ]:
df = pd.read_csv('../EXIST2021_dataset/training/EXIST2021_training.tsv', sep='\t')

df['task1_encoding'] = df['task1'].replace({'sexist': 1, 'non-sexist':  0})      # Codificamos la variable a num?rica

df.head()

* Separate the data for task 2

In [ ]:
df_t2 = df.drop(df.loc[df['task1']=='non-sexist'].index)

df_t2['task2_encoding'] = df_t2['task2'].replace({'ideological-inequality' : 0,
                                                  'stereotyping-dominance' : 1, 'misogyny-non-sexual-violence': 2,
                                                  'sexual-violence' : 3, 'objectification': 4})
df_t2.head()

* Process and clean the texts

In [ ]:
tweets_clean_token, tweets_stem = preprocess_tweets(df['text'], df['language'], keep_hastags=False)
tweets_clean = [' '.join(tw) for tw in tweets_clean_token]

In [ ]:
tweets_clean_token2, tweets_stem2 = preprocess_tweets(df_t2['text'], df_t2['language'], keep_hastags=False)
tweets_clean2 = [' '.join(tw) for tw in tweets_clean_token2]

---

## Modeling <a class="anchor" id="model"></a>

### Baseline <a class="anchor" id="baseline"></a>

In [ ]:
baseline_models = [svm.SVC(),
                   RandomForestClassifier(),
                   LogisticRegression(),
                   tree.DecisionTreeClassifier()]

#### Task1 <a class="anchor" id="b-t1"></a>

* IDF

In [ ]:
idf_vect, idf_matrix = codify_tfidf_vec(tweets_stem)
idf_df = pd.DataFrame(idf_matrix.toarray(), columns=idf_vect.get_feature_names())

In [ ]:
X = idf_matrix
y = df.task1_encoding

In [ ]:
for model in baseline_models:
    print(model)
    print('Evaluando...')
    model_acc, model_f1 = crossval_kfold(X, y, model, 5)
    print('Accuracy obtenido k-fold:', model_acc) 
    print('Accuracy medio:', sum(model_acc)/5, '\n')

*  Word embeddings 

In [ ]:
glove_twitter = api.load("glove-twitter-200")
w2v_data = word2vec.Word2Vec(tweets_clean_token, vector_size=200)

vectors_embeddings = get_vectors_embeddings(glove_twitter, w2v_data, tweets_clean_token)

In [ ]:
X = vectors_embeddings
y = df.task1_encoding

In [ ]:
import warnings
warnings.filterwarnings("ignore")
for model in baseline_models:
    print(model)
    print('Evaluando...')
    model_acc, model_f1 = crossval_kfold(X, y, model, 5)
    print('Accuracy obtenido k-fold:', model_acc) 
    print('Accuracy medio:', sum(model_acc)/5, '\n')

---

 #### Task2 <a class="anchor" id="b-t2"></a>

* IDF

In [ ]:
idf_vect2, idf_matrix2 = codify_tfidf_vec(tweets_stem2)
idf_df2 = pd.DataFrame(idf_matrix2.toarray(), columns=idf_vect2.get_feature_names())

In [ ]:
X = idf_matrix2
y = np.array((df_t2.task2_encoding))

In [ ]:
for model in baseline_models:
    print(model)
    print('Evaluando...')
    model_acc, model_f1 = crossval_kfold(X, y, model, 5)
    print('F1 macro obtenido k-fold:', model_f1) 
    print('F1 medio:', sum(model_f1)/5, '\n')

* Word embeddings

In [ ]:
w2v_data2 = word2vec.Word2Vec(tweets_clean_token2, vector_size=200)

vectors_embeddings2 = get_vectors_embeddings(glove_twitter, w2v_data2, tweets_clean_token2)

In [ ]:
X = vectors_embeddings2
y =  np.array((df_t2.task2_encoding))

In [ ]:
for model in baseline_models:
    print(model)
    print('Evaluando...')
    model_acc, model_f1 = crossval_kfold(X, y, model, 5)
    print('F1 macro obtenido k-fold:', model_f1) 
    print('F1 medio:', sum(model_f1)/5, '\n')

---

### Tunning parameters <a class="anchor" id="tunning"></a>

In [ ]:
svm_params = {'C': [0.1, 1, 10, 100], 
              'gamma': [1, 0.1, 0.01, 0.001],
              'kernel': ['rbf', 'poly', 'linear', 'sigmoid']}

rf_params = {'bootstrap': [True, False],
             'max_depth': [5, 10, 100, None],
             'criterion': ['gini', 'entropy']}

lr_params = {'penalty': ['l1','l2'], 
             'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

grid_svm = GridSearchCV(svm.SVC(), svm_params, 
                        refit = True, verbose = 2, n_jobs=12, scoring='accuracy')

grid_rf = GridSearchCV(RandomForestClassifier(), rf_params, 
                       refit = True, verbose = 2, n_jobs=12, scoring='accuracy')

grid_lr = GridSearchCV(LogisticRegression(), lr_params, 
                       refit = True, verbose = 2, n_jobs=12, scoring='accuracy')

#### Task1 <a class="anchor" id="tp-t1"></a>

* IDF

In [ ]:
X = idf_matrix
y = df.task1_encoding

In [ ]:
grid_svm.fit(X, y)
print(grid_svm.best_score_)
print(grid_svm.best_params_)

In [ ]:
grid_rf.fit(X, y)
print(grid_rf.best_score_)
print(grid_rf.best_params_)

In [ ]:
grid_lr.fit(X, y)
print(grid_lr.best_score_)
print(grid_lr.best_params_)

* Word embeddings

In [ ]:
X = vectors_embeddings
y = df.task1_encoding

In [ ]:
grid_svm.fit(X, y)
print(grid_svm.best_score_)
print(grid_svm.best_params_)

In [ ]:
grid_rf.fit(X, y)
print(grid_rf.best_score_)
print(grid_rf.best_params_)

In [ ]:
grid_lr.fit(X, y)
print(grid_lr.best_score_)
print(grid_lr.best_params_)

---

#### Task2 <a class="anchor" id="tp-t2"></a>

* IDF

In [ ]:
X = idf_matrix2
y = np.array((df_t2.task2_encoding))

In [ ]:
grid_svm.fit(X, y)
print(grid_svm.best_score_)
print(grid_svm.best_params_)

In [ ]:
grid_rf.fit(X, y)
print(grid_rf.best_score_)
print(grid_rf.best_params_)

In [ ]:
grid_lr.fit(X, y)
print(grid_lr.best_score_)
print(grid_lr.best_params_)

* Word embeddings

In [ ]:
X = vectors_embeddings2
y =  np.array((df_t2.task2_encoding))

In [ ]:
grid_svm.fit(X, y)
print(grid_svm.best_score_)
print(grid_svm.best_params_)

In [ ]:
grid_rf.fit(X, y)
print(grid_rf.best_score_)
print(grid_rf.best_params_)

In [ ]:
grid_lr.fit(X, y)
print(grid_lr.best_score_)
print(grid_lr.best_params_)

---

### Ensemble methods <a class="anchor" id="ensemble"></a>

#### Task1 <a class="anchor" id="e-t1"></a>

* IDF

In [ ]:
X = idf_matrix
y = df.task1_encoding

clf_svm = svm.SVC(C=1, gamma=1, kernel='sigmoid')
clf_lr = LogisticRegression(C=1, penalty='l2')
clf_rf = RandomForestClassifier(bootstrap=True, criterion='gini', max_depth= 100)

In [ ]:
bagging_svm = BaggingClassifier(base_estimator=clf_svm, n_estimators=10)
print('Acc. medio Bagging con SVM:', sum(crossval_kfold(X, y, bagging_svm, 5)[0])/5)

bagging_lr = BaggingClassifier(base_estimator=clf_lr, n_estimators=10)
print('Acc. medio Bagging con LR:', sum(crossval_kfold(X, y, bagging_lr, 5)[0])/5)

In [ ]:
#boost_rf = AdaBoostClassifier(base_estimator=clf_rf, n_estimators=100)
#print('Acc. medio AdaBoost con RF:', sum(crossval_kfold(X, y, boost_rf, 5)[0])/5)

boost_lr = AdaBoostClassifier(base_estimator=clf_lr, n_estimators=100)
print('Acc. medio AdaBoost con LR:', sum(crossval_kfold(X, y, boost_lr, 5)[0])/5)

In [ ]:
stack = StackingClassifier(estimators=[('svm', clf_svm)], final_estimator= clf_lr)
print('Acc. medio Stack SVM LR:', sum(crossval_kfold(X, y, stack, 5)[0])/5)

* Word embeddings

In [ ]:
X = vectors_embeddings
y = df.task1_encoding

clf_svm = svm.SVC(C=1, gamma=0.1, kernel='rbf')
clf_lr = LogisticRegression(C=0.1, penalty='l2')
clf_rf = RandomForestClassifier(bootstrap=True, criterion='entropy', max_depth= 100)

In [ ]:
bagging_svm = BaggingClassifier(base_estimator=clf_svm, n_estimators=10)
print('Acc. medio Bagging con SVM:', sum(crossval_kfold(X, y, bagging_svm, 5)[0])/5)

bagging_lr = BaggingClassifier(base_estimator=clf_lr, n_estimators=10)
print('Acc. medio Bagging con LR:', sum(crossval_kfold(X, y, bagging_lr, 5)[0])/5)

In [ ]:
#boost_rf = AdaBoostClassifier(base_estimator=clf_rf, n_estimators=100)
#print('Acc. medio AdaBoost con RF:', sum(crossval_kfold(X, y, boost_rf, 5)[0])/5)

boost_lr = AdaBoostClassifier(base_estimator=clf_lr, n_estimators=100)
print('Acc. medio AdaBoost con LR:', sum(crossval_kfold(X, y, boost_lr, 5)[0])/5)

In [ ]:
stack = StackingClassifier(estimators=[('svm', clf_svm)], final_estimator= clf_lr)
print('Acc. medio Stack SVM LR:', sum(crossval_kfold(X, y, stack, 5)[0])/5)

---

#### Task2 <a class="anchor" id="e-t2"></a>

* IDF

In [ ]:
X = idf_matrix2
y = np.array((df_t2.task2_encoding))

clf_svm = svm.SVC(C=10, gamma=0.1, kernel='rbf')
clf_lr = LogisticRegression(C=1, penalty='l2')
clf_rf = RandomForestClassifier(bootstrap=True, criterion='entropy', max_depth= 100)

In [ ]:
bagging_svm = BaggingClassifier(base_estimator=clf_svm, n_estimators=10)
print('F1 medio Bagging con SVM:', sum(crossval_kfold(X, y, bagging_svm, 5)[1])/5)

bagging_lr = BaggingClassifier(base_estimator=clf_lr, n_estimators=10)
print('F1 medio Bagging con LR:', sum(crossval_kfold(X, y, bagging_lr, 5)[1])/5)

In [ ]:
boost_lr = AdaBoostClassifier(base_estimator=clf_lr, n_estimators=100)
print('F1 medio AdaBoost con LR:', sum(crossval_kfold(X, y, boost_lr, 5)[1])/5)

In [ ]:
stack = StackingClassifier(estimators=[('svm', clf_svm)], final_estimator= clf_lr)
print('F1 medio Stack SVM LR:', sum(crossval_kfold(X, y, stack, 5)[1])/5)

* Word Embeddings

In [ ]:
X = vectors_embeddings2
y = np.array((df_t2.task2_encoding))

clf_svm = svm.SVC(C=10, gamma=0.01, kernel='rbf')
clf_lr = LogisticRegression(C=0.1, penalty='l2')
clf_rf = RandomForestClassifier(bootstrap=False, criterion='gini', max_depth= None)

In [ ]:
bagging_svm = BaggingClassifier(base_estimator=clf_svm, n_estimators=10)
print('F1 medio Bagging con SVM:', sum(crossval_kfold(X, y, bagging_svm, 5)[1])/5)

bagging_lr = BaggingClassifier(base_estimator=clf_lr, n_estimators=10)
print('F1 medio Bagging con LR:', sum(crossval_kfold(X, y, bagging_lr, 5)[1])/5)

In [ ]:
boost_lr = AdaBoostClassifier(base_estimator=clf_lr, n_estimators=100)
print('F1 medio AdaBoost con LR:', sum(crossval_kfold(X, y, boost_lr, 5)[1])/5)

In [ ]:
stack = StackingClassifier(estimators=[('svm', clf_svm)], final_estimator= clf_lr)
print('F1 medio Stack SVM LR:', sum(crossval_kfold(X, y, stack, 5)[1])/5)

---